In [1]:
from pathlib import Path
from dask.distributed import LocalCluster
import dask.dataframe as dd
import hvplot.dask

cluster = LocalCluster()  # Fully-featured local Dask cluster
client = cluster.get_client()
print(client)
cluster

<Client: 'tcp://127.0.0.1:54142' processes=4 threads=16, memory=128.00 GiB>


LocalCluster(5fb71e3b, 'tcp://127.0.0.1:54142', workers=4, threads=16, memory=128.00 GiB)

In [2]:
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from loky import get_reusable_executor
from pathlib import Path
from tqdm.notebook import tqdm
import numba

batch_size = 1600000
total = 96094530 // batch_size

def read_file_in_batches(fl, batch_size=batch_size):
    """
    Reads a file line by line in specified batch sizes.

    Args:
        filepath (str): The path to the file to read.
        batch_size (int): The number of lines to include in each batch.

    Yields:
        list: A list containing lines of the current batch.
    """
    current_batch = []
    for line in fl:
        current_batch.append(line.strip())  # .strip() removes newline characters
        if len(current_batch) == batch_size:
            yield current_batch
            current_batch = []
    # Yield any remaining lines in the last batch
    if current_batch:
        yield current_batch


# executor = get_reusable_executor(max_workers=16)
@numba.jit
def get_count(line: str):
    c = len(line.split(","))
    p = len(line.split("."))
    return c, p, line


csv_file = Path("../data/output_e1d.csv")
out_file = Path("../data/out.csv")
bad_lines = 0
with ThreadPoolExecutor(max_workers=160)  as executor:
    with csv_file.open("r") as csv, out_file.open("w") as out:
        for lines in tqdm(read_file_in_batches(csv), total=total):
            futures = [executor.submit(get_count, line) for line in lines]
            for res in as_completed(futures):
                res = res.result()
                if res[0] == 9 and res[1] == 6:
                    out.write(res[2] + "\n")
                else:
                    bad_lines = bad_lines + 1
print(bad_lines)

  0%|          | 0/60 [00:00<?, ?it/s]

113


In [3]:
names = [
    "electron_sector",
    "w",
    "q2",
    "theta",
    "phi",
    "mm2",
    "cut_fid",
    "helicty",
    "type",
]
dtype = {
        "electron_sector": "int8",
        "helicty": "int8",
        "w": "float32",
        "q2": "float32",
        "theta": "float32",
        "phi": "float32",
        "mm2": "float32",
        "cut_fid": "bool",
    }

csv_file = Path("../data/out.csv")

In [4]:
csv_data = dd.read_csv(
    csv_file, names=names, dtype=dtype, index_col=False, blocksize=25e6, sample=250
)

In [5]:
csv_data.head()

,electron_sector,w,q2,theta,phi,mm2,cut_fid,helicty,type
0,2,1.504319,2.243423,0.953701,3.118837,0.958821,True,-1,mc_rec
1,3,1.965226,1.347291,0.857256,5.036975,0.905967,True,1,mc_rec
2,4,1.678264,2.088247,2.254447,2.271331,1.205594,True,-1,mc_rec
3,1,1.435435,3.167328,0.459820,2.934700,0.897783,True,-1,mc_rec
4,4,1.284377,2.115568,1.494792,0.936972,0.883222,True,-1,mc_rec


In [6]:
# Plot
csv_data.hvplot.scatter(x="w", y="q2", rasterize=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'3d22d7ba-423c-4ee0-a3ce-c910157c754e': {'version…

In [7]:
csv_data.hvplot.kde("w")
    

/Users/tylern/github.com/tylern4/physics_code/.venv/lib/python3.13/site-packages/scipy/_lib/_util.py:1272: RuntimeWarning: divide by zero encountered in vecdot
  return np.vecdot(x1, x2, axis=axis)
/Users/tylern/github.com/tylern4/physics_code/.venv/lib/python3.13/site-packages/scipy/_lib/_util.py:1272: RuntimeWarning: overflow encountered in vecdot
  return np.vecdot(x1, x2, axis=axis)
/Users/tylern/github.com/tylern4/physics_code/.venv/lib/python3.13/site-packages/scipy/_lib/_util.py:1272: RuntimeWarning: invalid value encountered in vecdot
  return np.vecdot(x1, x2, axis=axis)


:Distribution   [w]   (Density)